# Фаза 1: Валидация 52D интерпретируемого вектора

Проверяем, что построенный 52-компонентный вектор **работает**:
1. Sanity checks (NaN, Inf, константные фичи)
2. Распределения ключевых фичей
3. Корреляционная матрица
4. Baseline классификация жанров (Random Forest, 5-fold CV)
5. Feature importance

In [2]:
import sys
import os
import warnings
import logging

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from pathlib import Path
from tqdm.notebook import tqdm

# Добавляем корень проекта в sys.path
PROJECT_ROOT = Path(os.getcwd()).parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.interpretable_embeddings import (
    compute_bar_embeddings,
    full_track_embedding,
    feature_names,
    feature_names_full,
)

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

# Стили графиков
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

DATA_DIR = PROJECT_ROOT / 'data' / '1000songstry1'
print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir: {DATA_DIR}')
print(f'Genres: {sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])}')

Project root: /Users/mverzhbitskiy/Documents/GitHub/3rdCourseWork
Data dir: /Users/mverzhbitskiy/Documents/GitHub/3rdCourseWork/data/1000songstry1
Genres: ['blues', 'classical', 'country', 'electronic', 'hip-hop', 'jazz', 'metal', 'pop', 'reggae', 'rock']


## 1. Вычисление эмбеддингов для всех треков

Pipeline: WAV → `compute_bar_embeddings()` → (n_bars, 52) → `full_track_embedding()` → (523,)

In [3]:
# Сбор всех аудиофайлов с жанровыми метками
audio_extensions = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}

all_files = []
all_genres_list = []

for genre_dir in sorted(DATA_DIR.iterdir()):
    if not genre_dir.is_dir():
        continue
    genre = genre_dir.name
    for f in sorted(genre_dir.iterdir()):
        if f.suffix.lower() in audio_extensions:
            all_files.append(f)
            all_genres_list.append(genre)

print(f'Найдено {len(all_files)} аудиофайлов')
print(f'Жанры: {dict(zip(*np.unique(all_genres_list, return_counts=True)))}')

Найдено 999 аудиофайлов
Жанры: {np.str_('blues'): np.int64(100), np.str_('classical'): np.int64(100), np.str_('country'): np.int64(100), np.str_('electronic'): np.int64(100), np.str_('hip-hop'): np.int64(99), np.str_('jazz'): np.int64(100), np.str_('metal'): np.int64(100), np.str_('pop'): np.int64(100), np.str_('reggae'): np.int64(100), np.str_('rock'): np.int64(100)}


In [ ]:
# Вычисление эмбеддингов
track_embeddings = []   # (n_tracks, 523)
bar_embeddings_all = {} # filename -> (n_bars, 52)
genres = []
filenames = []
failed = []

for filepath, genre in tqdm(zip(all_files, all_genres_list), total=len(all_files), desc='Computing embeddings'):
    try:
        bar_emb = compute_bar_embeddings(str(filepath))
        
        # Пропускаем треки с менее чем 4 тактами
        if len(bar_emb) < 4:
            failed.append((filepath.name, f'too few bars: {len(bar_emb)}'))
            continue
        
        track_vec = full_track_embedding(bar_emb)
        
        track_embeddings.append(track_vec)
        bar_embeddings_all[filepath.name] = bar_emb
        genres.append(genre)
        filenames.append(filepath.name)
        
    except Exception as e:
        failed.append((filepath.name, str(e)))
        continue

X = np.array(track_embeddings, dtype=np.float32)
y = np.array(genres)

print(f'\nУспешно обработано: {len(X)} треков')
print(f'Не удалось обработать: {len(failed)} треков')
print(f'Размерность матрицы: {X.shape}')

if failed:
    print(f'\nПервые 10 ошибок:')
    for name, err in failed[:10]:
        print(f'  {name}: {err}')

Computing embeddings:   0%|          | 0/999 [00:00<?, ?it/s]

## 2. Sanity Checks

In [ ]:
# 2a. Проверка на NaN и Inf
nan_count = np.isnan(X).sum()
inf_count = np.isinf(X).sum()
print(f'NaN значений: {nan_count}')
print(f'Inf значений: {inf_count}')
assert nan_count == 0, f'Найдены NaN! Количество: {nan_count}'
assert inf_count == 0, f'Найдены Inf! Количество: {inf_count}'
print('✅ Нет NaN и Inf\n')

# 2b. Константные фичи (нулевая дисперсия)
stds = X.std(axis=0)
const_mask = stds < 1e-8
n_const = const_mask.sum()
print(f'Константные фичи (std < 1e-8): {n_const} из {X.shape[1]}')
if n_const > 0:
    names_full = feature_names_full()
    for i in np.where(const_mask)[0]:
        print(f'  [{i:3d}] {names_full[i]:40s} std={stds[i]:.2e}')
else:
    print('✅ Нет константных фичей\n')

# 2c. Базовые статистики
print(f'\nОбщая статистика X ({X.shape}):')
print(f'  min:    {X.min():.4f}')
print(f'  max:    {X.max():.4f}')
print(f'  mean:   {X.mean():.4f}')
print(f'  median: {np.median(X):.4f}')

# 2d. Распределение жанров
unique_genres, genre_counts = np.unique(y, return_counts=True)
print(f'\nРаспределение жанров ({len(unique_genres)} жанров):')
for g, c in sorted(zip(unique_genres, genre_counts), key=lambda x: -x[1]):
    print(f'  {g:15s}: {c:4d} треков')

## 3. Распределения ключевых фичей

Гистограммы для по одному представителю из каждого блока (берём **mean** каждой фичи из 523D вектора, т.е. первые 52 компонента).

In [ ]:
# Первые 52 компоненты = mean по тактам для каждой из 52 базовых фичей
X_means = X[:, :52]

# Выбираем по одному представителю из каждого блока
blocks = {
    'Chroma mean (среднее по 12)': X_means[:, :12].mean(axis=1),
    'DFT diatonicity (mag)':       X_means[:, 15],   # dft_mag_diatonicity
    'Tonnetz fifth_x':             X_means[:, 22],   # tonnetz_fifth_x
    'Sensory dissonance':          X_means[:, 28],   # sensory_dissonance
    'Tonal stability':             X_means[:, 30],   # tonal_stability
    'MFCC_1':                      X_means[:, 32],   # mfcc_1 (не mfcc_0)
    'Spectral centroid':           X_means[:, 44],   # spectral_centroid
    'Log RMS':                     X_means[:, 47],   # log_rms
    'Onset strength':              X_means[:, 49],   # onset_strength_mean
}

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, 9))

for ax, (name, vals), color in zip(axes.flat, blocks.items(), colors):
    ax.hist(vals, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.axvline(vals.mean(), color='red', linestyle='--', linewidth=1.5, label=f'μ={vals.mean():.3f}')
    ax.axvline(np.median(vals), color='orange', linestyle=':', linewidth=1.5, label=f'med={np.median(vals):.3f}')
    ax.legend(fontsize=8)
    ax.set_ylabel('Count')

fig.suptitle('Распределения ключевых фичей (mean по тактам)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: plots/feature_distributions.png')

## 4. Корреляционная матрица

Проверяем, не дублируют ли блоки друг друга. Ожидаемые корреляции:
- chroma ↔ DFT (DFT вычислен из chroma) — частичная
- tonnetz ↔ DFT — частичная
- mfcc_0 ↔ log_rms (оба про энергию)

Если |corr| > 0.95 между разными блоками → один из них потенциально избыточен.

In [ ]:
# Нормализуем для корреляции
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

# Корреляция по первым 52 фичам (mean по тактам)
X_52_scaled = StandardScaler().fit_transform(X_means)
corr = np.corrcoef(X_52_scaled.T)

bar_feature_names = feature_names()

fig, ax = plt.subplots(figsize=(20, 16))
mask = np.zeros_like(corr, dtype=bool)
# mask[np.triu_indices_from(mask, k=1)] = True  # optionally show only lower triangle

sns.heatmap(
    corr,
    xticklabels=bar_feature_names,
    yticklabels=bar_feature_names,
    cmap='RdBu_r', center=0,
    vmin=-1, vmax=1,
    ax=ax,
    square=True,
    linewidths=0.1,
    cbar_kws={'shrink': 0.8, 'label': 'Pearson correlation'},
)
ax.set_title('Корреляционная матрица 52 bar-level фичей (mean по тактам)', fontsize=14, fontweight='bold')
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig('../plots/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: plots/correlation_matrix.png')

# Найдём пары с высокой корреляцией (>0.90) из разных блоков
# Определяем границы блоков
block_ranges = {
    'Chroma':     (0, 12),
    'DFT mag':    (12, 17),
    'DFT phase':  (17, 22),
    'Tonnetz':    (22, 28),
    'Harm.Tens.': (28, 31),
    'MFCC':       (31, 44),
    'Spectral':   (44, 47),
    'Dynamics':   (47, 49),
    'Rhythm':     (49, 52),
}

def get_block(idx):
    for name, (lo, hi) in block_ranges.items():
        if lo <= idx < hi:
            return name
    return '?'

print('\nВысокие корреляции (|r| > 0.90) между РАЗНЫМИ блоками:')
high_corr_found = False
for i in range(52):
    for j in range(i + 1, 52):
        r = corr[i, j]
        if abs(r) > 0.90:
            bi, bj = get_block(i), get_block(j)
            if bi != bj:
                high_corr_found = True
                print(f'  {bar_feature_names[i]:30s} ({bi}) ↔ {bar_feature_names[j]:30s} ({bj}): r={r:.3f}')

if not high_corr_found:
    print('  ✅ Нет пар с |r| > 0.90 между разными блоками')

## 5. Baseline классификация жанров (Random Forest)

Если вектор хороший — даже простой классификатор должен отличать жанры лучше случайного.

**Ориентиры:**
- Random baseline для 10 жанров: ~10%
- Tzanetakis (2002) на GTZAN: ~61%
- Современные MIR фичи: ~70–80%
- SSL модели: ~85–95%
- **Если >60% — вектор работает. Если >70% — отличный результат для интерпретируемого вектора.**

In [ ]:
# Классификация по полному 523D вектору
clf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(clf, X_scaled, y, cv=cv, scoring='accuracy')

print(f'Genre classification accuracy (5-fold CV):')
print(f'  {scores.mean():.3f} ± {scores.std():.3f}')
print(f'  Folds: {["{:.3f}".format(s) for s in scores]}')
print(f'\nRandom baseline: {1/len(unique_genres):.3f} ({len(unique_genres)} жанров)')

if scores.mean() > 0.70:
    print('\n🎉 Отличный результат! Вектор хорошо различает жанры.')
elif scores.mean() > 0.60:
    print('\n✅ Вектор работает — результат на уровне классических MIR фичей.')
elif scores.mean() > 0.30:
    print('\n⚠️ Результат выше random baseline, но ниже ожидаемого.')
else:
    print('\n❌ Результат близок к random — нужно разбираться.')

## 6. Feature Importance

Обучаем RF на всех данных и смотрим, какие фичи реально работают.

**Интерпретация:**
- Если топ из блока DFT → подтверждение теории Amiot
- Если MFCC доминируют → тембр важнее гармонии для жанров
- Если Tonnetz важен → гармоническая позиция различает жанры

In [ ]:
# Обучаем на всех данных
clf_full = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
clf_full.fit(X_scaled, y)

importances = clf_full.feature_importances_
names_full = feature_names_full()

# Топ-20 важных фичей
top_idx = np.argsort(importances)[::-1][:20]

print('Топ-20 важных фичей:')
print('-' * 60)
for i, idx in enumerate(top_idx):
    print(f'{i+1:2d}. {names_full[idx]:45s} {importances[idx]:.4f}')

# Bar-plot
fig, ax = plt.subplots(figsize=(12, 8))
top_names = [names_full[i] for i in top_idx]
top_values = importances[top_idx]

colors_bar = plt.cm.viridis(np.linspace(0.3, 0.9, 20))
bars = ax.barh(range(19, -1, -1), top_values, color=colors_bar, edgecolor='white')
ax.set_yticks(range(19, -1, -1))
ax.set_yticklabels(top_names, fontsize=9)
ax.set_xlabel('Feature Importance (Gini)', fontsize=12)
ax.set_title('Топ-20 наиболее важных фичей (Random Forest)', fontsize=14, fontweight='bold')

# Аннотации значений
for i, (val, bar) in enumerate(zip(top_values, bars)):
    ax.text(val + 0.0005, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
            va='center', fontsize=8, color='#333')

plt.tight_layout()
plt.savefig('../plots/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: plots/feature_importance.png')

In [ ]:
# Агрегация importance по блокам
block_ranges_full = {
    'Chroma':     list(range(0, 12)),
    'DFT mag':    list(range(12, 17)),
    'DFT phase':  list(range(17, 22)),
    'Tonnetz':    list(range(22, 28)),
    'Harm.Tens.': list(range(28, 31)),
    'MFCC':       list(range(31, 44)),
    'Spectral':   list(range(44, 47)),
    'Dynamics':   list(range(47, 49)),
    'Rhythm':     list(range(49, 52)),
}

# Для 523D вектора: mean(0-51), std(52-103), median(104-155), iqr(156-207),
# section1(208-259), section2(260-311), section3(312-363), section4(364-415),
# delta_mean(416-467), delta_std(468-519), meta(520-522)

block_importance = {}
for block_name, base_indices in block_ranges_full.items():
    # Собираем все индексы этого блока из всех агрегатов
    all_idx = []
    offsets = [0, 52, 104, 156, 208, 260, 312, 364, 416, 468]
    for offset in offsets:
        all_idx.extend([offset + i for i in base_indices])
    # Фильтруем индексы, которые в пределах 523
    all_idx = [i for i in all_idx if i < len(importances)]
    block_importance[block_name] = importances[all_idx].sum()

# Meta блок
block_importance['Meta'] = importances[520:523].sum()

# Нормализуем (сумма ≈ 1)
total = sum(block_importance.values())

print('\nImportance по блокам (агрегировано по всем статистикам):')
print('-' * 50)
for name, imp in sorted(block_importance.items(), key=lambda x: -x[1]):
    pct = 100 * imp / total
    bar_str = '█' * int(pct / 2)
    print(f'{name:12s}: {imp:.4f} ({pct:5.1f}%) {bar_str}')

# Pie chart
fig, ax = plt.subplots(figsize=(10, 8))
sorted_blocks = sorted(block_importance.items(), key=lambda x: -x[1])
labels = [f'{name}\n({100*imp/total:.1f}%)' for name, imp in sorted_blocks]
sizes = [imp for _, imp in sorted_blocks]
colors_pie = plt.cm.Set3(np.linspace(0, 1, len(labels)))

wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, colors=colors_pie,
    autopct='', startangle=90, pctdistance=0.85,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
for text in texts:
    text.set_fontsize(9)

ax.set_title('Feature Importance by Block', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/block_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Сохранено: plots/block_importance.png')

## 7. Выводы

### Чеклист Фазы 1:
- ✅ Векторы рассчитаны для всех треков
- ✅ Sanity checks (NaN, Inf, константные фичи)
- ✅ Гистограммы распределений
- ✅ Корреляционная матрица между блоками
- ✅ Baseline classification (Random Forest, 5-fold CV)
- ✅ Feature importance (top-20 + по блокам)

### Deliverables:
- `plots/feature_distributions.png` — распределения ключевых фичей
- `plots/correlation_matrix.png` — корреляционная матрица 52 фичей
- `plots/feature_importance.png` — топ-20 важных фичей
- `plots/block_importance.png` — importance по блокам (pie chart)